# T04 — Self-attention y máscara causal

## 1. Título y paper

**Paper:** *Attention Is All You Need* (Vaswani et al., 2017)  
**Fuente primaria:** [arXiv:1706.03762](https://arxiv.org/abs/1706.03762)  
**Foco de esta miniatura:** atender a la propia secuencia, y no atender al futuro  
**Ficha completa:** [`P08_transformer`](../../papers/foundational/P08_transformer/README.md)


## 2. Objetivos

1. Distinguir self-attention de cross-attention.
2. Implementar la máscara causal y comprobar que impide ver el futuro.


## 3. Prerrequisitos

- Python 3.11+ con el paquete instalado (`pip install -e .`).
- Notebook [`P08_transformer`](P08_transformer.ipynb) al menos hojeado.
- Álgebra de vectores: producto escalar, norma y softmax.


## 4. Intuición

Self-attention es que cada palabra pregunte al resto de **su propia** frase. La máscara causal es taparle los ojos hacia adelante: si el modelo va a generar el token siguiente, no puede haberlo visto ya.


## 5. Concepto mínimo

```text
self-attention  : Q, K, V salen de la MISMA secuencia
cross-attention : Q del decoder, K y V del encoder
máscara causal  : score_ij = −∞ para j > i  →  α_ij = 0
```


## 6. Código explicado

Código mínimo, sin dependencias externas.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
from ai_evolution.papers_lab import scaled_dot_product_attention

X = [[1.0, 0.0], [0.9, 0.1], [0.0, 1.0], [0.1, 0.9]]
libre = scaled_dot_product_attention(X, X, X)
causal = scaled_dot_product_attention(X, X, X, causal=True)
print('sin máscara:')
for fila in libre['weights']:
    print('  ', [round(w, 3) for w in fila])
print('con máscara causal:')
for fila in causal['weights']:
    print('  ', [round(w, 3) for w in fila])

## 7. Predicción antes de ejecutar

¿Qué forma tendrá la matriz enmascarada? ¿Cuánto sumará la primera fila?

> Escribe tu respuesta antes de continuar.


## 8. Experimento controlado


In [ ]:
for i, fila in enumerate(causal['weights']):
    futuro = sum(fila[i + 1:])
    print(f'fila {i}: suma={sum(fila):.6f} · masa sobre el futuro={futuro:.6f}')

## 9. Salida interpretable

Matriz triangular inferior: la masa sobre el futuro es exactamente 0 y cada fila sigue sumando 1. La posición 0 solo puede atenderse a sí misma, por eso su peso es 1,0.


## 10. Comentario pedagógico

Esta miniatura aísla **una** pieza del bloque. Aislar es didáctico y también es una simplificación: en el modelo real todas las piezas interactúan y se entrenan juntas.


## 11. Error o anti-patrón deliberado


In [ ]:
print('Error frecuente: aplicar la máscara DESPUÉS del softmax.')
import math
def softmax(zs):
    m = max(zs)
    e = [math.exp(z - m) for z in zs]
    return [v / sum(e) for v in e]
p = softmax([2.0, 1.0, 3.0])
p_mal = [p[0], p[1], 0.0]
print('tras poner a cero el futuro:', [round(v, 4) for v in p_mal], '· suma =', round(sum(p_mal), 4))
print('→ ya no suma 1: la distribución quedó rota')

## 12. Corrección


In [ ]:
p_bien = softmax([2.0, 1.0, -1e9])
print('máscara ANTES del softmax:', [round(v, 6) for v in p_bien], '· suma =', round(sum(p_bien), 6))

## 13. Desafío guiado

Construye la máscara de padding (ignorar tokens de relleno) y comprueba que es independiente de la causal.


## 14. Desafío autónomo

Reescribe esta pieza con proyecciones aprendidas y comprueba que tu implementación reproduce las propiedades verificadas aquí (sumas, formas, invariantes). Documenta la semilla.


## 15. Evidencia de aprendizaje

Guarda la salida del experimento, tu predicción previa y una frase sobre qué invariante acabas de verificar.


## 16. Cierre

Pieza cubierta: **atender a la propia secuencia, y no atender al futuro**. Ya puede describirse con precisión, sin metáforas.


## 17. Conexión con el siguiente hito

Una sola cabeza captura un tipo de relación. El paper usa varias en paralelo (T05).
